In [ ]:
%load_ext autoreload
%autoreload all

In [ ]:
# Standard Library imports
from pathlib import Path
from typing import Optional

# External imports
import fiftyone as fo
from fiftyone.core.dataset import Dataset, load_dataset
import fiftyone.brain as fob
from fiftyone.core.view import DatasetView

In [ ]:
def list_all_images(dir_path: Path):
    """ """
    suffixes = {".png", ".jpg", ".jpeg"}
    return [p.resolve() for p in dir_path.rglob("*") if p.suffix.lower() in suffixes]


def add_samples_to_dataset(
    dataset: Dataset | DatasetView,
    sources: Path | str | list[Path],
    tags: Optional[list[str]] = None,
):
    """ """

    if isinstance(sources, (Path, str)):
        sources = [Path(sources)]
    if isinstance(sources, list):
        sources = [Path(s) for s in sources]

    for source in sources:
        files = list_all_images(source)

        samples = []
        for filepath in files:
            samples.append(fo.Sample(filepath=filepath, tags=tags))

        dataset.add_samples(samples)

In [ ]:
ROOT_PATH = Path(r"FILL_ME")
FIFTYONE_DATASET_NAME = "fish_crops_RGBA"

In [ ]:
try:
    dataset = Dataset(FIFTYONE_DATASET_NAME, persistent=True, overwrite=False)
except ValueError:
    print("Loading dataset")
    dataset = load_dataset(FIFTYONE_DATASET_NAME, create_if_necessary=False)

In [ ]:
add_samples_to_dataset(dataset, [ROOT_PATH], tags=[])

In [ ]:
# When auto=False is provided, a new App window is created only when you call session.show():
session = fo.launch_app(dataset, browser="chrome", auto=False)

# Now open http://localhost:5151

In [ ]:
brain_key_similarity = "similarity"

if brain_key_similarity in dataset.list_brain_runs():
    print("Loading existen similarity index")
    similarity_index = dataset.load_brain_results(brain_key_similarity)
else:
    model_name = "clip-vit-base32-torch"

    similarity_index = fob.compute_similarity(
        samples=dataset, model=model_name, brain_key=brain_key_similarity
    )

In [ ]:
# Use the computed near duplicates index with a different threshold to search again for duplicates
similarity_index.find_duplicates(thresh=0.5)  # Higher thresh, more duplicates shown
session.view = similarity_index.duplicates_view()

In [ ]:
# Generate a 2D visualization of near duplicates
brain_key_visualization = "visualization"
viz_results = fob.compute_visualization(
    dataset, brain_key=brain_key_visualization, similarity_index=similarity_index
)